# K-Nearest Neighbors: Learning by Similarity

**K-Nearest Neighbors (KNN)** is one of the simplest and most intuitive machine learning algorithms. It makes predictions based on the similarity of a new data point to its K closest neighbors in the training set. No model training is required — the "learning" happens entirely at prediction time.

In this notebook we will cover:
- The intuition behind KNN
- Distance metrics and their impact
- How to choose the optimal K
- Distance weighting schemes
- KNN for classification and regression
- Why feature scaling is critical
- The curse of dimensionality
- Algorithm variants (brute force, KD Tree, Ball Tree)
- Decision boundary visualization
- Comparison with other classifiers

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_iris, load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

print('All imports successful.')

---
## 2. The Intuition: "Tell Me Who Your Friends Are…"

KNN is motivated by a simple idea: **similar things are close to each other**.

If a new, unknown data point is surrounded by many points of the same class, it likely belongs to that class. If its nearest neighbors have a certain numerical value, the new point probably has a similar value.

Key characteristics:
- **Non-parametric**: no underlying assumption about data distribution
- **Instance-based** (lazy learner): no explicit training phase; the entire training set is stored and used at prediction time
- **Memory-based**: predictions rely entirely on proximity in feature space

---
## 3. How KNN Works

Given a query point $x_q$:
1. Compute the distance from $x_q$ to every point in the training set
2. Select the K points with the smallest distance
3. For **classification**: take a majority vote among the K neighbors
4. For **regression**: take the average (or weighted average) of the neighbors' values

The algorithm has three critical design choices:
- **K**: the number of neighbors
- **Distance metric**: how we measure "closeness"
- **Weighting**: whether all neighbors vote equally or by proximity

---
## 4. Distance Metrics

The choice of distance metric defines the geometry of the feature space. Scikit-learn's KNN supports the following:

| Metric | Formula | When to use |
|--------|---------|-------------|
| **Euclidean** | $\sqrt{\sum_{i=1}^n (x_i - y_i)^2}$ | Continuous, well-scaled features (default) |
| **Manhattan** | $\sum_{i=1}^n |x_i - y_i|$ | High-dimensional or grid-like spaces |
| **Minkowski** | $(\sum_{i=1}^n |x_i - y_i|^p)^{1/p}$ | Generalization (p=2 → Euclidean, p=1 → Manhattan) |
| **Hamming** | $\frac{1}{n}\sum_{i=1}^n \mathbb{1}[x_i \neq y_i]$ | Binary/categorical features |
| **Cosine** | $1 - \frac{x \cdot y}{\|x\|\|y\|}$ | Text or high-dimensional sparse data |

In [ ]:
def compute_distances(x, y):
    """Compute various distance metrics between two 1D arrays."""
    return {
        'Euclidean': np.sqrt(np.sum((x - y) ** 2)),
        'Manhattan': np.sum(np.abs(x - y)),
        'Minkowski (p=3)': np.power(np.sum(np.power(np.abs(x - y), 3)), 1/3),
        'Cosine': 1 - np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)),
        'Chebyshev': np.max(np.abs(x - y))
    }

np.random.seed(42)
a = np.random.randn(5)
b = np.random.randn(5)

dists = compute_distances(a, b)
for name, val in dists.items():
    print(f'{name:>20s}: {val:.4f}')

### Visualizing Distance Metrics

Let's see how different distance metrics change the shape of neighborhoods around a point.

In [ ]:
from sklearn.metrics import DistanceMetric

xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
center = np.array([0, 0])

metrics = {
    'Euclidean (p=2)': 'euclidean',
    'Manhattan (p=1)': 'manhattan',
    'Minkowski (p=0.5)': 'minkowski',
    'Chebyshev (p=inf)': 'chebyshev'
}

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, (title, metric) in zip(axes.ravel(), metrics.items()):
    if metric == 'minkowski':
        dist = DistanceMetric.get_metric(metric, p=0.5)
        z = dist.pairwise(np.c_[xx.ravel(), yy.ravel()], center.reshape(1, -1))
        z = z.reshape(xx.shape)
    elif metric == 'chebyshev':
        z = np.maximum(np.abs(xx), np.abs(yy))
    else:
        dist = DistanceMetric.get_metric(metric)
        z = dist.pairwise(np.c_[xx.ravel(), yy.ravel()], center.reshape(1, -1))
        z = z.reshape(xx.shape)
    cs = ax.contourf(xx, yy, z, levels=20, cmap='viridis')
    ax.plot(0, 0, 'ro', markersize=10)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_aspect('equal')
    plt.colorbar(cs, ax=ax, shrink=0.8)

plt.suptitle('Unit Circles Under Different Distance Metrics', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Load and Explore the Iris Dataset

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['species'] = pd.Categorical.from_codes(y, target_names)

print(f'Shape: {X.shape}')
print(f'Classes: {target_names}')
print(f'Samples per class: {pd.Series(y).value_counts().to_dict()}')
df.head()

In [ ]:
sns.pairplot(df, hue='species', height=2.5, palette='Set2')
plt.suptitle('Iris Dataset — Pairplot', y=1.02, fontsize=14)
plt.show()

---
## 6. The Effect of K — Bias-Variance Tradeoff

| Small K | Large K |
|---------|--------|
| Decision boundary is very flexible (high variance) | Decision boundary is smooth (low variance) |
| Sensitive to noise / outliers | Resistant to noise |
| **Overfitting** risk | **Underfitting** risk |
| Low bias | High bias |

Choosing K is the single most important hyperparameter in KNN.

### Finding Optimal K — Elbow Method on Error Rate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

k_range = range(1, 51)
train_errors = []
test_errors = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_errors.append(1 - knn.score(X_train_scaled, y_train))
    test_errors.append(1 - knn.score(X_test_scaled, y_test))

best_k = k_range[np.argmin(test_errors)]
print(f'Best K (lowest test error): {best_k} (error rate = {min(test_errors):.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, train_errors, 'o-', label='Training Error', markersize=4, color='steelblue')
ax.plot(k_range, test_errors, 's-', label='Test Error', markersize=4, color='crimson')
ax.axvline(best_k, color='green', linestyle='--', alpha=0.7, label=f'Best K = {best_k}')
ax.set_xlabel('K (Number of Neighbors)', fontsize=12)
ax.set_ylabel('Error Rate', fontsize=12)
ax.set_title('K vs Error Rate — Elbow Method', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(range(0, 51, 5))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Distance Weighting: Uniform vs Distance-Based

By default, KNN uses **uniform** weighting — every neighbor gets one vote. But we can weight votes by the **inverse of distance**, so closer neighbors have more influence:

- `weights='uniform'`: all K neighbors contribute equally
- `weights='distance'`: votes are weighted by $1 / d$ where $d$ is the distance to the query point
- Custom: any user-provided callable

In [ ]:
weight_schemes = ['uniform', 'distance']
results = {}

for w in weight_schemes:
    knn = KNeighborsClassifier(n_neighbors=15, weights=w)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    results[w] = scores
    print(f'weights="{w}" -> CV accuracy: {scores.mean():.4f} (±{scores.std():.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x_pos = np.arange(len(weight_schemes))
for i, w in enumerate(weight_schemes):
    ax.bar(i, results[w].mean(), yerr=results[w].std(), capsize=8,
           color=['steelblue', 'coral'][i], alpha=0.8, width=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(weight_schemes, fontsize=11)
ax.set_ylabel('Cross-Validation Accuracy', fontsize=11)
ax.set_title('Uniform vs Distance Weighting', fontsize=13, fontweight='bold')
ax.set_ylim(0.7, 1.0)
ax.axhline(results['uniform'].mean(), color='steelblue', linestyle='--', alpha=0.5)
ax.axhline(results['distance'].mean(), color='coral', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 8. KNN for Regression

`KNeighborsRegressor` predicts the **mean (or weighted mean)** of the K nearest neighbors' target values.

This works well for smoothly varying functions but struggles with extrapolation (it cannot predict values beyond the range seen in training).

In [ ]:
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.randn(80) * 0.1

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

fig, ax = plt.subplots(figsize=(10, 5))
X_plot = np.linspace(0, 5, 500).reshape(-1, 1)

for k, color, style in zip([1, 5, 20], ['crimson', 'green', 'navy'], ['-', '--', ':']):
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_reg_train, y_reg_train)
    y_plot = knn_reg.predict(X_plot)
    ax.plot(X_plot, y_plot, color=color, linestyle=style, linewidth=2,
            label=f'K = {k} (R² = {r2_score(y_reg_test, knn_reg.predict(X_reg_test)):.3f})')

ax.scatter(X_reg_train, y_reg_train, color='steelblue', s=30, alpha=0.6, label='Training data')
ax.scatter(X_reg_test, y_reg_test, color='orange', s=40, alpha=0.8, marker='^', label='Test data')
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('KNN Regression with Different K Values', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 9. Feature Scaling is CRITICAL for KNN

KNN relies on **distance calculations**. If one feature has a much larger range (e.g., salary in $ vs age in years), it will dominate the distance and the other features become irrelevant.

**Always** standardize or normalize features before applying KNN.

In [ ]:
np.random.seed(42)
X_unscaled = np.column_stack([
    np.random.randn(150) * 0.5 + 5,      # feature 1: small range ~[3,7]
    np.random.randn(150) * 100 + 500     # feature 2: large range ~[200,800]
])
y_unscaled = (X_unscaled[:, 0] + X_unscaled[:, 1] / 200 + np.random.randn(150) * 0.3 > 6).astype(int)

X_us_train, X_us_test, y_us_train, y_us_test = train_test_split(
    X_unscaled, y_unscaled, test_size=0.3, random_state=42
)

scaler_std = StandardScaler()
X_us_scaled = scaler_std.fit_transform(X_us_train)
X_us_test_scaled = scaler_std.transform(X_us_test)

knn_no_scale = KNeighborsClassifier(n_neighbors=5)
knn_no_scale.fit(X_us_train, y_us_train)
acc_no = knn_no_scale.score(X_us_test, y_us_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_us_scaled, y_us_train)
acc_scaled = knn_scaled.score(X_us_test_scaled, y_us_test)

print(f'Accuracy WITHOUT scaling: {acc_no:.4f}')
print(f'Accuracy WITH scaling:    {acc_scaled:.4f}')
print(f'Improvement:              {acc_scaled - acc_no:+.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

xx, yy = np.meshgrid(np.linspace(2, 8, 200), np.linspace(100, 900, 200))

for ax, (scaler_name, X_tr, X_te) in zip(
    axes,
    [
        ('No Scaling', X_us_train, X_us_test),
        ('StandardScaler', X_us_scaled, X_us_test_scaled)
    ]
):
    knn_viz = KNeighborsClassifier(n_neighbors=5)
    knn_viz.fit(X_tr, y_us_train)
    Z = knn_viz.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    scatter = ax.scatter(X_te[:, 0], X_te[:, 1], c=y_us_test, cmap='coolwarm',
                         edgecolor='k', s=40)
    ax.set_title(f'{scaler_name}\nTest Acc: {knn_viz.score(X_te, y_us_test):.3f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1 (small range)', fontsize=10)
    ax.set_ylabel('Feature 2 (large range)', fontsize=10)

plt.suptitle('KNN: With vs Without Feature Scaling', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. The Curse of Dimensionality

As the number of dimensions increases, **all points become far apart** from each other. The ratio of the nearest-point distance to the farthest-point distance approaches 1 — meaning KNN cannot distinguish neighbors from non-neighbors.

Consequences:
- Distances lose meaning
- You need exponentially more training data to fill the space
- KNN performance degrades rapidly

In [ ]:
np.random.seed(42)
dimensions = [1, 2, 3, 5, 10, 20, 50, 100]
ratio_results = []

for d in dimensions:
    points = np.random.randn(500, d)
    dists = np.linalg.norm(points[1:] - points[0], axis=1)
    ratio = dists.min() / dists.max()
    ratio_results.append(ratio)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(dimensions, ratio_results, 'o-', color='darkred', linewidth=2, markersize=8)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='d_min / d_max = 0.5')
ax.set_xlabel('Number of Dimensions', fontsize=12)
ax.set_ylabel('d_min / d_max', fontsize=12)
ax.set_title('Curse of Dimensionality — Distance Ratios Collapse', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xscale('log')
ax.set_xticks(dimensions)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
plt.tight_layout()
plt.show()

print('As dimensions increase, all points become nearly equidistant.')

---
## 11. Algorithm Options: Brute Force, KD Tree, Ball Tree

Scikit-learn provides three algorithms for finding nearest neighbors:

| Algorithm | Training Time | Prediction Time | Memory | Best for |
|-----------|--------------|----------------|--------|----------|
| **Brute Force** | O(1) | O(nd) | O(n) | Small datasets (< 1000 rows) |
| **KD Tree** | O(n log n) | O(log n) per query | O(n) | Low dimensions (< 20), structured data |
| **Ball Tree** | O(n log n) | O(log n) per query | O(n) | High dimensions (> 20), clustered data |

You can set `algorithm='auto'` and scikit-learn will choose the best one based on the data.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import time

X_big, y_big = make_classification(n_samples=5000, n_features=10, n_informative=5,
                                    n_redundant=2, random_state=42)
X_big_tr, X_big_te, y_big_tr, y_big_te = train_test_split(X_big, y_big, test_size=0.3, random_state=42)

algorithms = ['brute', 'kd_tree', 'ball_tree']
timings = {}

for algo in algorithms:
    knn = KNeighborsClassifier(n_neighbors=5, algorithm=algo, leaf_size=30)
    t0 = time.perf_counter()
    knn.fit(X_big_tr, y_big_tr)
    t1 = time.perf_counter()
    knn.predict(X_big_te[:100])
    t2 = time.perf_counter()
    timings[algo] = {'fit': t1 - t0, 'predict': t2 - t1}
    print(f'{algo:>12s}: fit={t1-t0:.4f}s, predict(100)={t2-t1:.4f}s')

---
## 12. KNN Classification — Full Pipeline

Now let's apply KNN on the Iris dataset with proper scaling, cross-validation, and evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn_final = KNeighborsClassifier(n_neighbors=best_k, weights='distance', algorithm='auto')
knn_final.fit(X_train, y_train)
y_pred = knn_final.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print(f'KNN (K={best_k}, distance-weighted) Test Accuracy: {test_acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=target_names))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Confusion Matrix — KNN on Iris', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 13. Decision Boundary Visualization for Different K

Let's visualize how K controls the complexity of the decision boundary.

In [ ]:
X_pca = PCA(n_components=2, random_state=42).fit_transform(iris.data)
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_pca, y, test_size=0.3, random_state=42, stratify=y
)

scaler_pca = StandardScaler()
Xp_train = scaler_pca.fit_transform(Xp_train)
Xp_test = scaler_pca.transform(Xp_test)

h = 0.02
x_min, x_max = Xp_train[:, 0].min() - 1, Xp_train[:, 0].max() + 1
y_min, y_max = Xp_train[:, 1].min() - 1, Xp_train[:, 1].max() + 1
xx_d, yy_d = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

k_values = [1, 3, 5, 15]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#00AA00', '#0000FF'])

for k, ax in zip(k_values, axes.ravel()):
    knn_viz = KNeighborsClassifier(n_neighbors=k)
    knn_viz.fit(Xp_train, yp_train)
    Z = knn_viz.predict(np.c_[xx_d.ravel(), yy_d.ravel()])
    Z = Z.reshape(xx_d.shape)
    ax.contourf(xx_d, yy_d, Z, cmap=cmap_light, alpha=0.8)
    ax.scatter(Xp_train[:, 0], Xp_train[:, 1], c=yp_train, cmap=cmap_bold,
               edgecolor='k', s=40)
    acc = knn_viz.score(Xp_test, yp_test)
    ax.set_title(f'K = {k}  (Test Acc = {acc:.3f})', fontsize=13, fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('Decision Boundaries for Different K Values', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 14. Pros and Cons of KNN

| ✅ Pros | ❌ Cons |
|---------|--------|
| Simple and intuitive | Slow at prediction (no training, but expensive inference) |
| No training phase (lazy learner) | Sensitive to irrelevant / unscaled features |
| Works for both classification and regression | Suffers from curse of dimensionality |
| Non-parametric — no distribution assumptions | Requires storing the entire training set |
| Naturally multi-class | Choosing K requires cross-validation |
| Decision boundaries can be arbitrarily complex | Struggles with imbalanced data |

---
## 15. Comparison: KNN vs Logistic Regression vs Decision Tree

Let's compare these three fundamentally different classifiers on the Wine dataset.

In [ ]:
wine = load_wine()
X_w, y_w = wine.data, wine.target
Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X_w, y_w, test_size=0.3, random_state=42, stratify=y_w
)

s_w = StandardScaler()
Xw_train_s = s_w.fit_transform(Xw_train)
Xw_test_s = s_w.transform(Xw_test)

models = {
    'KNN (K=5)': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42)
}

results_comp = []
for name, model in models.items():
    model.fit(Xw_train_s, yw_train)
    train_acc = model.score(Xw_train_s, yw_train)
    test_acc = model.score(Xw_test_s, yw_test)
    results_comp.append({'Model': name, 'Train Acc': train_acc, 'Test Acc': test_acc})
    print(f'{name:>22s}: Train = {train_acc:.4f}, Test = {test_acc:.4f}')

In [ ]:
df_comp = pd.DataFrame(results_comp)
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(df_comp))
width = 0.35
ax.bar(x_pos - width/2, df_comp['Train Acc'], width, label='Train', color='steelblue', alpha=0.8)
ax.bar(x_pos + width/2, df_comp['Test Acc'], width, label='Test', color='coral', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(df_comp['Model'], fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Model Comparison on Wine Dataset', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0.6, 1.05)
for i, row in df_comp.iterrows():
    ax.text(i - width/2, row['Train Acc'] + 0.01, f'{row["Train Acc"]:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.text(i + width/2, row['Test Acc'] + 0.01, f'{row["Test Acc"]:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 16. Practice Exercises

Test your understanding of KNN with these exercises:

### Exercise 1: KNN on a New Dataset
Load the `digits` dataset from `sklearn.datasets.load_digits()`. Apply KNN with optimal K and report the accuracy.

### Exercise 2: Weighted vs Uniform Voting
Using the Wine dataset, compare KNN with `weights='uniform'` vs `weights='distance'` across K = 1 to 30. Plot both curves.

### Exercise 3: Impact of Feature Scaling
On the Breast Cancer dataset (`load_breast_cancer`), compare KNN accuracy with:
- No scaling
- StandardScaler
- MinMaxScaler
- RobustScaler

### Exercise 4: Custom Distance Metric
Implement KNN with a custom distance function (e.g., Canberra distance) using the `metric` parameter with a callable, and compare performance on the Iris dataset.

### Exercise 5: KNN for Regression with Grid Search
Use `GridSearchCV` with `KNeighborsRegressor` on the California housing dataset (`fetch_california_housing`) to find the best combination of `n_neighbors`, `weights`, and `p` (Minkowski exponent).

In [ ]:
# === Exercise 1: KNN on Digits Dataset ===
from sklearn.datasets import load_digits

# Your code here
pass

In [ ]:
# === Exercise 2: Weighted vs Uniform Voting ===

# Your code here
pass

In [ ]:
# === Exercise 3: Impact of Feature Scaling ===
from sklearn.datasets import load_breast_cancer

# Your code here
pass

In [ ]:
# === Exercise 4: Custom Distance Metric ===

# Your code here
pass

In [ ]:
# === Exercise 5: KNN Regression with Grid Search ===
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import fetch_california_housing

# Your code here
pass

---
## Summary

In this notebook we covered:
- **KNN intuition**: prediction by similarity
- **Distance metrics**: Euclidean, Manhattan, Minkowski, Chebyshev, Cosine
- **Effect of K**: small K = overfitting, large K = underfitting → elbow method
- **Distance weighting**: uniform vs distance-based voting
- **KNN for regression**: `KNeighborsRegressor` with synthetic data
- **Feature scaling**: critical for KNN — demonstrated with/without scaling
- **Curse of dimensionality**: distances become meaningless in high dimensions
- **Algorithms**: brute force, KD Tree, Ball Tree
- **Decision boundaries**: visualized for K = 1, 3, 5, 15
- **Comparison**: KNN vs Logistic Regression vs Decision Tree on Wine dataset

KNN is a powerful baseline algorithm. Its simplicity makes it an excellent first choice, but its computational cost at prediction time and sensitivity to irrelevant features and high dimensions must be carefully considered.